Zinhal
======

### Import

In [ ]:
import os
import numpy as np
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import chess
from chess import pgn
from tqdm import tqdm

## Data preprocessing

### Loading data

In [4]:
def get_number_of_games(file_path):
    number_of_games = 0
    with open(file_path, 'r') as pgn_file:
        while True:
            if not pgn.skip_game(pgn_file):
                break
            number_of_games += 1
    return number_of_games
    

def load_pgn(file_path, offset):
    games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="r+")
    with open(file_path, 'r') as pgn_file:
        i = offset
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games[i] = game
            i += 1
    del games
    return i

files = [file for file in os.listdir("../lib/data/pgn") if file.endswith(".pgn")]
LIMIT_OF_FILES = min(len(files), 30)
number_of_games = 0
for file in tqdm(files[:LIMIT_OF_FILES]):
    number_of_games += get_number_of_games(f"../lib/data/pgn/{file}")

games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="w+", shape=(number_of_games))
del games
offset = 0
for file in tqdm(files[:LIMIT_OF_FILES]):
    offset = load_pgn(f"../lib/data/pgn/{file}", offset)
games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="r+")

100%|██████████| 30/30 [03:16<00:00,  6.55s/it]


In [5]:
print(f"Games parsed: {len(games)}")

Games parsed: 247364


### Convert data into tensors

In [6]:
from ridoc import generate_eval_lables

In [7]:
positions, results = generate_eval_lables(games)
print(f"Number of samples: {len(results)}")

100%|██████████| 247364/247364 [31:45<00:00, 129.80it/s]

Number of samples: 19796735


In [28]:
positions.flush()
results.flush()
positions = np.memmap(filename="../lib/data/npy/eval_position.npy", dtype=np.float32, mode="r")
results = np.memmap(filename="../lib/data/npy/reults.npy", dtype=np.float32, mode="r")

## Preliminary actions

In [8]:
from jesinia import EvalDataset
from violet import EvalModel

In [29]:
dataset = EvalDataset(positions, results)

dataloader = DataLoader(dataset, batch_size=32, pin_memory=True)

loader = DataLoader(dataset, batch_size=32, pin_memory=True)
data_iter = iter(loader)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = EvalModel().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adadelta(model.parameters(), lr=0.005)

Using device: cuda


## Traning

In [ ]:
num_epochs = 1

model_name = f"1_2_{LIMIT_OF_FILES}_{num_epochs}"

next_batch = data_iter.next() # start loading the first batch
next_batch = [ _.cuda(non_blocking=True) for _ in next_batch ]  # with pin_memory=True and non_blocking=True, this will copy data to GPU non blockingly

for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    with tqdm(dataloader) as iterator:
        for inputs, labels in iterator:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)

            outputs = model(inputs)

            loss = criterion(outputs, labels)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            running_loss += loss.item()
    end_time = time.time()
    epoch_time = end_time - start_time
    minutes: int = int(epoch_time // 60)
    seconds: int = int(epoch_time) - minutes * 60
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {running_loss / len(dataloader):.4f}, Time: {minutes}m{seconds}s")
    torch.save({'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict
                }, f"../models/checkpoints/e{model_name}.pth")

 22%|██▏       | 135343/618648 [04:19<15:27, 521.05it/s]


KeyboardInterrupt: 

### Save the model

In [ ]:
torch.save(model, f"../models/e{model_name}.pth")

positions.flush()
results.flush()